# 02 · What is *rank*?

**Quick recap from notebook 01:** a layer is just `output = W @ x`, and every output
is built as a **mix of `W`'s columns**. We ended on a question:

> *How many genuinely **different** directions does a matrix have?*

That number is called the **rank**, and this notebook is all about it — because it's
the single idea that makes LoRA possible. Tiny numbers, as always. Let's go!

In [ ]:
import numpy as np
np.random.seed(0)

## Step 1 — "Different directions" vs "the same direction twice"

A column of a matrix is a **direction** (from notebook 01). The question is whether
the columns point in *truly different* directions, or whether some are just copies.

- **Matrix A:** columns `[1,0]` and `[0,1]` — these point two different ways. **2 real directions.**
- **Matrix B:** columns `[1,2]` and `[2,4]` — but `[2,4]` is just `2 × [1,2]`! It's the
  *same* direction, scaled. So really there's only **1 real direction.**

In [ ]:
A = np.array([[1, 0],
              [0, 1]])     # two different directions
B = np.array([[1, 2],
              [2, 4]])     # 2nd column = 2 x the 1st  -> same direction

print("A: truly different columns -> rank", np.linalg.matrix_rank(A))
print("B: 2nd col is 2x the 1st   -> rank", np.linalg.matrix_rank(B))

**That's the whole definition:**

> **rank = the number of genuinely independent directions in a matrix.**

`A` has rank 2. `B` *looks* like a 2×2 matrix with 4 numbers, but it only has rank 1 —
it secretly does a 1-direction job.

## Step 2 — "Low rank" means the matrix does a *smaller job* than it looks

Here's a 3×3 matrix where the **3rd column = 1st + 2nd column** (so it adds nothing new).
Its rank is 2, not 3. To *prove* it only does a 2-D job, we'll throw **200 different
inputs** at it and check: the outputs can never escape a flat 2-D plane.

In [ ]:
W = np.array([[1., 0, 1],
              [2, 1, 3],
              [1, 1, 2]])      # column 2 = column 0 + column 1

print("rank of W:", np.linalg.matrix_rank(W))

X = np.random.randn(3, 200)    # 200 totally different inputs
outputs = W @ X
print("rank of all 200 outputs:", np.linalg.matrix_rank(outputs),
      " -> every output trapped in a 2-D plane")

So **low rank = redundant = wasted dimensions.** Keep this feeling: a low-rank matrix
is "smaller than it looks." In a moment we'll bet that the *change* we make when
fine-tuning is exactly this kind of small-looking matrix.

## Step 3 — The simplest possible matrix: one direction (a "rank-1" piece)

What's the smallest building block? A matrix made from **a single direction**. We take
one direction `u` and say how much of it to place in each column, using a row `v`.
Multiplying them (`np.outer`) is called an **outer product**:

In [ ]:
u = np.array([1, 2])         # ONE direction (a column of 2 numbers)
v = np.array([1, 0, -1])     # how much of u to put in each of 3 columns

M = np.outer(u, v)           # a 2x3 matrix built from just ONE direction
print("M =\n", M)
print("rank of M:", np.linalg.matrix_rank(M), " (just 1 direction)")

Every column of `M` is a multiple of `u` — one direction only — so **rank 1**. This is
the atom we build everything out of.

## Step 4 — Stack `r` of these pieces → and that stack *is* `B · A`

Now stack **2** single-direction pieces. Beautifully, "stack 2 directions" is the exact
same thing as one **tall-skinny** matrix times one **short-wide** matrix:

- `B` holds the directions, as its **columns**  → shape `(rows, r)`
- `A` holds how to use them, as its **rows**     → shape `(r, cols)`

The number of pieces we stacked is called **`r`** (you'll also hear it called the
**rank** of the update). Here `r = 2`.

In [ ]:
B = np.array([[1, 0],
              [2, 1]])       # 2 directions, as columns  -> shape (2, r=2)
A = np.array([[1, 0, -1],
              [0, 3,  0]])   # how to use each direction  -> shape (r=2, 3)

BA = B @ A
print("B @ A =\n", BA)
print("rank:", np.linalg.matrix_rank(BA), " (= r = 2)")

# It's literally the sum of the two single-direction pieces from Step 3:
piece1 = np.outer(B[:, 0], A[0])
piece2 = np.outer(B[:, 1], A[1])
print("equals piece1 + piece2 ?", np.allclose(BA, piece1 + piece2))

## Recap

- **rank** = number of genuinely independent directions in a matrix.
- **low rank** = redundant = the matrix does a smaller job than its size suggests.
- Any rank-`r` matrix can be built by **stacking `r` single-direction pieces**, which is
  exactly a product **`B · A`** where `B` has `r` columns and `A` has `r` rows.
- **`r` is a number we get to choose** — how many directions we allow.

The big tease: storing a matrix as `B · A` uses *way* fewer numbers than the full matrix
when `r` is small. We'll cash that in soon.

**BAM!** Next: **`03 — training, and why the change to `W` is small`.**